In [0]:
from pathlib import Path
from typing import Optional
from databricks.sdk.runtime import *
from pyspark.sql import functions as F, Window

def read_sqlserver_with_secrets(
    *,
    scope: str = "jdbc",
    user_key: str = "sql_user",
    password_key: str = "sql_password",
    server: str = "awsu1-10wp6d01p.amer.thermo.com",
    port: int = 1433,
    database: str = "MIDB",
    query: Optional[str] = None,
    table: Optional[str] = None,
    prepare_query: Optional[str] = None,
    encrypt: bool = True,
    trust_server_certificate: bool = True,
    driver: str = "com.microsoft.sqlserver.jdbc.SQLServerDriver",
    cache: bool = False,
    dbutils=None,
    show_shape: bool = True,
    **jdbc_options,
):
    sql_user = dbutils.secrets.get(scope=scope, key=user_key)
    sql_password = dbutils.secrets.get(scope=scope, key=password_key)

    jdbc_url = (
        f"jdbc:sqlserver://{server}:{port};"
        f"databaseName={database};"
        f"encrypt={'true' if encrypt else 'false'};"
        f"trustServerCertificate={'true' if trust_server_certificate else 'false'};"
        f"user={sql_user};"
        f"password={sql_password};"
    )

    reader = spark.read.format("jdbc").option("url", jdbc_url).option("driver", driver)

    if query and table:
        raise ValueError("Provide only one of `query` or `table`, not both.")
    if not query and not table:
        raise ValueError("You must provide either `query` or `table`.")

    if prepare_query:
        reader = reader.option("prepareQuery", prepare_query)

    if query:
        reader = reader.option("query", query)
    else:
        reader = reader.option("dbtable", table)

    for k, v in jdbc_options.items():
        reader = reader.option(k, v)

    df = reader.load()

    if cache:
        df.cache()

    if show_shape:
        nrows = df.count()
        ncols = len(df.columns)
        print(f"Rows: {nrows}, Columns: {ncols}")

    return df

source_prepare = """
                DECLARE @Today date = CAST(GETDATE() AS date);
                DECLARE @usagethreshold dec = 0.8;
                DECLARE @weeksofsupplythreshold int = 4;
                
                """

source_query = Path("/Workspace/Users/kevin.wise@thermofisher.com/inventory_rebalance/sql/inventory_move_candidates.sql").read_text()

source_df = read_sqlserver_with_secrets(
    query=source_query,
    prepare_query=source_prepare,
    dbutils=dbutils,
    cache=True
)


destination_prepare = """
DECLARE @Today date = CAST(GETDATE() AS date);
DECLARE @usagethreshold dec = 0.8;"""

destination_query = Path("/Workspace/Users/kevin.wise@thermofisher.com/inventory_rebalance/sql/inventory_destinations.sql").read_text()

destination_df = read_sqlserver_with_secrets(
    query=destination_query,
    prepare_query=destination_prepare,
    dbutils=dbutils,
    cache=True
)

dst_dir = "dbfs:/Volumes/lsg/pre_dm/tfsdl_lsg_ops_prod/inventory_rebalance"
# -------- 5 Save df_features as Parquet to the volume -------- #
dbutils = globals().get("dbutils") or __import__("IPython").get_ipython().user_ns["dbutils"]
source_path = f"{dst_dir}/source_data"

(
    source_df
        .write
        .mode("overwrite")      # or "append", depending on your use case
        .parquet(source_path)
)

destination_path = f"{dst_dir}/source_data"

(
    source_df
        .write
        .mode("overwrite")      # or "append", depending on your use case
        .parquet(destination_path)
)